In [1]:
import os
import json
import random
import itertools
import math
import warnings
from tqdm.auto import tqdm
from tabulate import tabulate

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
from IPython.display import display, HTML

from models import get_model
from data import get_dataset
from utils.plot import *
from xai import *
from metrics import *


# plt.rcParams["font.size"] = 24configs/tweet-sentiment
# plt.rcParams["font.family"] = "cmr10"

os.environ["TOKENIZERS_PARALLELISM"] = "true"


[nltk_data] Downloading package punkt_tab to /home/xia/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
device = "cuda:6"
model_dir = "/home/xia/claim-xai/configs/imdb/base"
model_config = json.load( open(os.path.join(model_dir, "config.json"), "r") )
model_metadata = json.load( open(os.path.join(model_dir, "masker/_metadata.json"), "r") )

In [3]:
ds_train, ds_val, ds_test, num_classes = get_dataset(
    model_config["dataset_name"], 
    data_dir = "../_datasets", 
    splits = ["train", "validation", "test"],
    **model_config["dataset_args"], 
)


model = get_model(
    model_name = model_config["model_name"],
    checkpoint_path = os.path.join(model_dir, model_metadata["best_checkpoint"]),
    **model_config["model_args"]
).eval().to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


---

In [11]:
text, label = ds_test[0]

with torch.no_grad():
    inputs = model.preprocess(text)
    logits, weights = model(inputs)
pred = torch.argmax(logits).item()
print(f"pred={pred} label={label}")

tokens = inputs["input_ids"].cpu().squeeze(0)
tokens = tokens.tolist()


inputs = (inputs["input_ids"], inputs["attention_mask"])
input_embeds = model.text_encoder.embed(inputs[0])


explainer_our = OurAttribution(model)
attr_our = explainer_our(inputs)[0]

explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0] * 0))
attr_lig = explainer_lig(inputs[0], target=pred, additional_forward_args=inputs[1])[0]

explainer_saliency = SaliencyAttribution(model)
attr_saliency = explainer_saliency(input_embeds, target=pred, additional_forward_args=inputs[1])[0]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    explainer_dl = DeepLiftAttribution(model, baselines=(input_embeds * 0.0))
    attr_dl = explainer_dl(input_embeds, target=pred, additional_forward_args=inputs[1])[0]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    baselines_dlshap = torch.cat([ model.text_encoder.embed(model.preprocess(ds_train[i][0])["input_ids"]) for i in range(20) ], dim=0)
    explainer_dlshap = DeepLiftShapAttribution(model, baselines=baselines_dlshap)
    attr_dlshap = explainer_dlshap(input_embeds, target=pred, additional_forward_args=inputs[1].unsqueeze(1))[0]

for attr in [ attr_our, attr_lig, attr_saliency, attr_dl, attr_dlshap ]:
    attr_plot = attr.mean(dim=2)[0]
    attr_plot = (attr_plot - attr_plot.min()) / (attr_plot.max() - attr_plot.min())
    attr_plot = attr_plot.tolist()
    display_text_attribution(tokens[1:], attr_plot[1:], model.text_encoder._model.tokenizer, "<pad>")


pred=1 label=1


---

In [ ]:
def _accum_metrics(metrics, model_base, inputs, attribution):
    metrics["avg-drop"].accumulate(model_base, inputs, attribution)
    metrics["delete-auc"].accumulate(model_base, inputs, attribution)
    metrics["complexity"].accumulate(attribution)
    metrics["sparsity"].accumulate(attribution)

In [ ]:
model_base = get_model_wrapper(model)

methods = [ "ours", "layer-ig", "saliency", "deeplift", "deeplift-shap" ]
metrics = {
    method: {
        "avg-drop": AverageDropMetric(device),
        "delete-auc": DeletionAUCMetric(device),
        "complexity": ComplexityMetric(device),
        "sparsity": SparsityMetric(device)
    } for method in methods
}


# for i in tqdm(range(len(ds_test))):
for i in tqdm(range(500)):
    text, label = ds_test[i]

    with torch.no_grad():
        inputs = model.preprocess(text)
        logits, weights = model(inputs)
    pred = torch.argmax(logits).item()
    # print(f"pred={pred} label={label}")

    # tokens = inputs["input_ids"].cpu().squeeze(0)
    # tokens = tokens.tolist()


    inputs = (inputs["input_ids"], inputs["attention_mask"])
    input_embeds = model.text_encoder.embed(inputs[0])


    explainer_our = OurAttribution(model)
    attr_our = explainer_our(inputs)[0]
    _accum_metrics(metrics["ours"], model_base, inputs, attr_our)

    explainer_lig = LayerIntegratedGradientsAttribution(model, baselines=(inputs[0] * 0))
    attr_lig = explainer_lig(inputs[0], target=pred, additional_forward_args=inputs[1])[0]
    _accum_metrics(metrics["layer-ig"], model_base, inputs, attr_lig)

    explainer_saliency = SaliencyAttribution(model)
    attr_saliency = explainer_saliency(input_embeds, target=pred, additional_forward_args=inputs[1])[0]
    _accum_metrics(metrics["saliency"], model_base, inputs, attr_saliency)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        explainer_dl = DeepLiftAttribution(model, baselines=(input_embeds * 0.0))
        attr_dl = explainer_dl(input_embeds, target=pred, additional_forward_args=inputs[1])[0]
        _accum_metrics(metrics["deeplift"], model_base, inputs, attr_dl)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        baselines_dlshap = torch.cat([ model.text_encoder.embed(model.preprocess(ds_train[i][0])["input_ids"]) for i in range(20) ], dim=0)
        explainer_dlshap = DeepLiftShapAttribution(model, baselines=baselines_dlshap)
        attr_dlshap = explainer_dlshap(input_embeds, target=pred, additional_forward_args=inputs[1].unsqueeze(1))[0]
        _accum_metrics(metrics["deeplift-shap"], model_base, inputs, attr_dlshap)


  0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
tab_out = []
for method in metrics:
    out = [ method ]
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        out.append( f"{mean:.2f} +/- {std:.2f}" )
    tab_out.append( out )

print(tabulate(
    tab_out, 
    headers = ["method", *metrics["ours"]]
))

method         avg-drop       delete-auc     complexity         sparsity
-------------  -------------  -------------  -----------------  ----------------
ours           0.10 +/- 0.29  0.69 +/- 0.62  17.24 +/- 3.95     1.56 +/- 0.91
layer-ig       0.78 +/- 1.83  2.19 +/- 1.48  277.72 +/- 154.76  3.82 +/- 4.12
saliency       3.59 +/- 1.38  3.43 +/- 1.30  18.35 +/- 6.27     119.78 +/- 89.65
deeplift       0.17 +/- 0.78  1.83 +/- 1.00  312.61 +/- 113.76  2.41 +/- 1.31
deeplift-shap  0.17 +/- 0.83  1.79 +/- 0.97  315.93 +/- 107.08  2.35 +/- 1.33


In [ ]:
json_out = {}
for method in metrics:
    json_out[method] = {}
    for metric in metrics[method]:
        mean, std = metrics[method][metric].compute()
        json_out[method][metric] = { "mean": mean, "std": std }


In [ ]:
json.dump(json_out, open(os.path.join(model_dir, "metrics.json"), "w"), indent=4)

In [ ]:
"""
## Tweet-sentiment
method         avg-drop       delete-auc     complexity         sparsity
-------------  -------------  -------------  -----------------  -----------------
layer-ig       0.36 +/- 0.89  1.62 +/- 1.50  251.52 +/- 136.74  3.82 +/- 3.06
saliency       1.51 +/- 1.48  1.97 +/- 1.55  12.79 +/- 2.80     432.97 +/- 162.65
deeplift       0.09 +/- 0.40  1.07 +/- 1.03  309.86 +/- 99.49   2.32 +/- 1.16
deeplift-shap  0.08 +/- 0.38  1.00 +/- 0.93  313.58 +/- 85.08   2.20 +/- 0.84
ours           0.17 +/- 0.24  0.50 +/- 0.41  7.27 +/- 3.25      4.98 +/- 7.16



## Politifact
method         avg-drop       delete-auc     complexity         sparsity
-------------  -------------  -------------  -----------------  ----------------
layer-ig       0.06 +/- 0.22  0.46 +/- 0.54  295.45 +/- 113.58  2.59 +/- 1.46
saliency       0.81 +/- 0.83  0.76 +/- 0.79  16.58 +/- 5.42     113.37 +/- 94.45
deeplift       0.05 +/- 0.19  0.40 +/- 0.44  312.07 +/- 115.61  2.44 +/- 1.34
deeplift-shap  0.05 +/- 0.19  0.40 +/- 0.43  310.95 +/- 107.91  2.43 +/- 1.54
ours           0.24 +/- 0.33  0.46 +/- 0.50  2.58 +/- 1.32      89.54 +/- 112.53


IMDB
method         avg-drop       delete-auc     complexity         sparsity
-------------  -------------  -------------  -----------------  ----------------
layer-ig       0.78 +/- 1.83  2.19 +/- 1.48  277.72 +/- 154.76  3.82 +/- 4.12
saliency       3.59 +/- 1.38  3.43 +/- 1.30  18.35 +/- 6.27     119.78 +/- 89.65
deeplift       0.17 +/- 0.78  1.83 +/- 1.00  312.61 +/- 113.76  2.41 +/- 1.31
deeplift-shap  0.17 +/- 0.83  1.79 +/- 0.97  315.93 +/- 107.08  2.35 +/- 1.33
ours           0.10 +/- 0.29  0.69 +/- 0.62  17.24 +/- 3.95     1.56 +/- 0.91
"""

'\n## Tweet-sentiment\nmethod         avg-drop       delete-auc     complexity         sparsity\n-------------  -------------  -------------  -----------------  -----------------\nours           0.17 +/- 0.24  0.50 +/- 0.41  7.27 +/- 3.25      4.98 +/- 7.16\nlayer-ig       0.36 +/- 0.89  1.62 +/- 1.50  251.52 +/- 136.74  3.82 +/- 3.06\nsaliency       1.51 +/- 1.48  1.97 +/- 1.55  12.79 +/- 2.80     432.97 +/- 162.65\ndeeplift       0.09 +/- 0.40  1.07 +/- 1.03  309.86 +/- 99.49   2.32 +/- 1.16\ndeeplift-shap  0.08 +/- 0.38  1.00 +/- 0.93  313.58 +/- 85.08   2.20 +/- 0.84\n\n\n\n## Politifact\nmethod         avg-drop       delete-auc     complexity         sparsity\n-------------  -------------  -------------  -----------------  ----------------\nours           0.24 +/- 0.33  0.46 +/- 0.50  2.58 +/- 1.32      89.54 +/- 112.53\nlayer-ig       0.06 +/- 0.22  0.46 +/- 0.54  295.45 +/- 113.58  2.59 +/- 1.46\nsaliency       0.81 +/- 0.83  0.76 +/- 0.79  16.58 +/- 5.42     113.37 +/- 94.45\nd

---

In [ ]:
# shap_weights = attribution[0].mean(dim=2)[0]
# shap_weights = (shap_weights - shap_weights.min()) / (shap_weights.max() - shap_weights.min())
# shap_weights = shap_weights.tolist()

# display_text_attribution(tokens[1:], shap_weights[1:], model.text_encoder._model.tokenizer, "<pad>")

NameError: name 'attribution' is not defined

---